In [1]:
# Script: GenerarCsvBruto_Completo.py
import os
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm



def generar_csv_completo(CARPETA_TRAIN = "brain-1h-over-375day",ARCHIVO_SALIDA = "brain_bruto_1h.csv"):
    if not os.path.exists(CARPETA_TRAIN):
        print(f"❌ La carpeta '{CARPETA_TRAIN}' no existe.")
        return

    ficheros = sorted([f for f in os.listdir(CARPETA_TRAIN) if f.endswith(".xml")])
    print(f"📂 Encontrados {len(ficheros)} archivos XML. Iniciando proceso...")

    # Namespace del XML detectado anteriormente
    ns = {'ns': 'http://sndlib.zib.de/network'}
    
    datos_temporales = []

    # Bucle principal con barra de progreso
    for f in tqdm(ficheros, desc="Procesando XMLs"):
        ruta_completa = os.path.join(CARPETA_TRAIN, f)
        try:
            tree = ET.parse(ruta_completa)
            root = tree.getroot()
            
            # 1. Extraer Tiempo (tag <time> dentro de <meta>)
            tag_time = root.find(".//ns:time", ns)
            ts = tag_time.text if tag_time is not None else f.replace(".xml", "")
            
            fila = {'timestamp': ts}
            
            # 2. Extraer todas las demandas individuales
            for demand in root.findall(".//ns:demand", ns):
                src = demand.find("ns:source", ns).text
                dst = demand.find("ns:target", ns).text
                val_elem = demand.find("ns:demandValue", ns)
                
                if val_elem is not None:
                    nombre_enlace = f"{src}_{dst}"
                    fila[nombre_enlace] = float(val_elem.text)
            
            datos_temporales.append(fila)
            
        except Exception as e:
            # Si un archivo falla, continuamos con el siguiente
            continue

    print(f"\n📊 Creando DataFrame gigante (esto consumirá RAM)...")
    df = pd.DataFrame(datos_temporales)
    
    # 3. Limpieza y Ordenación
    print("🧹 Ordenando datos por tiempo...")
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y%m%d-%H%M', errors='coerce')
    df = df.sort_values('timestamp')
    
    # Reordenar columnas para que timestamp sea la primera
    cols = ['timestamp'] + sorted([c for c in df.columns if c != 'timestamp'])
    df = df[cols]
    
    print(f"💾 Guardando archivo CSV: {ARCHIVO_SALIDA}...")
    # Usamos index=False para no añadir la columna de IDs de pandas
    df.to_csv(ARCHIVO_SALIDA, index=False)
    
    print("-" * 40)
    print(f"✅ ¡PROCESO FINALIZADO!")
    print(f"Filas (instantes de tiempo): {len(df)}")
    print(f"Columnas (enlaces): {len(df.columns) - 1}")
    print(f"Archivo generado: {ARCHIVO_SALIDA}")
    print("-" * 40)

if __name__ == "__main__":
    generar_csv_completo(CARPETA_TRAIN = "directed-abilene-zhang-5min-over-6months-ALL",ARCHIVO_SALIDA = "abilene_bruto_5min.csv")

📂 Encontrados 48096 archivos XML. Iniciando proceso...


Procesando XMLs: 100%|██████████| 48096/48096 [00:42<00:00, 1123.85it/s]



📊 Creando DataFrame gigante (esto consumirá RAM)...
🧹 Ordenando datos por tiempo...
💾 Guardando archivo CSV: abilene_bruto_5min.csv...
----------------------------------------
✅ ¡PROCESO FINALIZADO!
Filas (instantes de tiempo): 48096
Columnas (enlaces): 132
Archivo generado: abilene_bruto_5min.csv
----------------------------------------


In [2]:
import pandas as pd
import os

def agrupar_por_nodos(archivo_entrada, archivo_salida):
    if not os.path.exists(archivo_entrada):
        print(f"❌ No existe el archivo: {archivo_entrada}")
        return

    print(f"📖 Cargando cabecera de {archivo_entrada}...")
    df_sample = pd.read_csv(archivo_entrada, nrows=1)
    todas_las_columnas = [c for c in df_sample.columns if c != 'timestamp']

    # Mapa Abilene: nos quedamos con las 4 primeras letras (código de ciudad)
    # 'ATLAM5_CHINng' -> 'ATLA_CHIN' ; 'ATLAng_CHINng' -> 'ATLA_CHIN'
    # Solo fusiona los dos routers de Atlanta (ATLAM5 y ATLAng); el resto solo cambia de nombre
    mapa_agrupacion = {}
    for col in todas_las_columnas:
        partes = col.split('_')
        if len(partes) == 2:
            mapa_agrupacion[col] = f"{partes[0][:4]}_{partes[1][:4]}"

    print(f"🧩 Mapeo completado. Flujos detectados: {len(set(mapa_agrupacion.values()))}")

    print("🚀 Iniciando agregación...")
    chunks = pd.read_csv(archivo_entrada, chunksize=1000)
    lista_df_agrupados = []

    for i, chunk in enumerate(chunks):
        ts = chunk['timestamp'].reset_index(drop=True)
        chunk_datos = chunk.drop(columns=['timestamp'])

        # min_count=1: si todas las componentes son NaN, el resultado sigue siendo NaN (no 0)
        chunk_agrupado = chunk_datos.T.groupby(mapa_agrupacion).sum(min_count=1).T.reset_index(drop=True)

        chunk_agrupado.insert(0, 'timestamp', ts)
        lista_df_agrupados.append(chunk_agrupado)
        print(f"   Bloque {i+1} procesado...")

    print("🔗 Uniendo bloques...")
    df_final = pd.concat(lista_df_agrupados, ignore_index=True)

    print(f"💾 Guardando en {archivo_salida}...")
    df_final.to_csv(archivo_salida, index=False)

    print("-" * 30)
    print(f"✅ ¡TERMINADO!")
    print(f"Archivo: {archivo_salida}")
    print(f"Columnas finales: {len(df_final.columns)}")
    print("-" * 30)

if __name__ == "__main__":
    agrupar_por_nodos("abilene_bruto_5min.csv", "abilene_CORE_5min.csv")

📖 Cargando cabecera de abilene_bruto_5min.csv...
🧩 Mapeo completado. Flujos detectados: 111
🚀 Iniciando agregación...
   Bloque 1 procesado...
   Bloque 2 procesado...
   Bloque 3 procesado...
   Bloque 4 procesado...
   Bloque 5 procesado...
   Bloque 6 procesado...
   Bloque 7 procesado...
   Bloque 8 procesado...
   Bloque 9 procesado...
   Bloque 10 procesado...
   Bloque 11 procesado...
   Bloque 12 procesado...
   Bloque 13 procesado...
   Bloque 14 procesado...
   Bloque 15 procesado...
   Bloque 16 procesado...
   Bloque 17 procesado...
   Bloque 18 procesado...
   Bloque 19 procesado...
   Bloque 20 procesado...
   Bloque 21 procesado...
   Bloque 22 procesado...
   Bloque 23 procesado...
   Bloque 24 procesado...
   Bloque 25 procesado...
   Bloque 26 procesado...
   Bloque 27 procesado...
   Bloque 28 procesado...
   Bloque 29 procesado...
   Bloque 30 procesado...
   Bloque 31 procesado...
   Bloque 32 procesado...
   Bloque 33 procesado...
   Bloque 34 procesado...
   Bloq

In [4]:
import pandas as pd
import numpy as np

def analizar_archivo(ruta_csv, freq="5min"):
    print(f"\n🔍 Analizando: {ruta_csv}")
    print("-" * 40)

    try:
        df = pd.read_csv(ruta_csv, parse_dates=['timestamp'])

        total_nans = df.isna().sum().sum()
        filas_vacias = df.drop(columns=['timestamp']).isna().all(axis=1).sum()
        columnas_con_nans = df.columns[df.isna().any()].tolist()

        print(f"✅ Total de celdas en el archivo: {df.size}")
        print(f"⚠️ Total de NaNs encontrados: {total_nans} ({df.drop(columns=['timestamp']).isna().values.mean():.2%})")
        print(f"🚫 Filas completamente vacías: {filas_vacias}")
        print(f"📊 Número de columnas: {len(df.columns)}")
        print(f"📈 Número de filas: {len(df)}")

        # NUEVO: instantes que faltan en la rejilla (filas que no existen)
        rejilla = pd.date_range(df['timestamp'].min(), df['timestamp'].max(), freq=freq)
        faltan = rejilla.difference(df['timestamp'])
        print(f"🕳️ Periodo: {df['timestamp'].min()} → {df['timestamp'].max()}")
        print(f"🕳️ Instantes que faltan: {len(faltan)} de {len(rejilla)} ({len(faltan)/len(rejilla):.2%})")
        if len(faltan):
            # agrupar los que faltan en huecos consecutivos y mostrar los más largos
            s = pd.Series(1, index=faltan)
            grupo = (s.index.to_series().diff() != pd.Timedelta(freq)).cumsum()
            huecos = s.groupby(grupo.values).agg(lambda x: x.size)
            inicios = s.index.to_series().groupby(grupo.values).first()
            top = pd.DataFrame({"inicio": inicios, "pasos": huecos}).sort_values("pasos", ascending=False).head(5)
            top["duracion"] = top["pasos"] * pd.Timedelta(freq)
            print("   Huecos más largos:")
            print(top[["inicio", "duracion"]].to_string(index=False))

        if total_nans > 0:
            print("\n📍 Detalle de NaNs por columna (Top 10):")
            detalles = df[columnas_con_nans].isna().sum().sort_values(ascending=False).head(10)
            print(detalles)
        else:
            print("\n💎 ¡Perfecto! El archivo no tiene valores nulos.")

    except Exception as e:
        print(f"❌ Error al leer el archivo: {e}")

if __name__ == "__main__":
    analizar_archivo("abilene_CORE_5min.csv", freq="5min")


🔍 Analizando: abilene_CORE_5min.csv
----------------------------------------
✅ Total de celdas en el archivo: 5386752
⚠️ Total de NaNs encontrados: 91525 (1.71%)
🚫 Filas completamente vacías: 0
📊 Número de columnas: 112
📈 Número de filas: 48096
🕳️ Periodo: 2004-03-01 00:00:00 → 2004-09-10 23:55:00
🕳️ Instantes que faltan: 7776 de 55872 (13.92%)
   Huecos más largos:
    inicio duracion
2004-03-15  18 days
2004-04-16   6 days
2004-04-29   2 days
2004-08-20   1 days

📍 Detalle de NaNs por columna (Top 10):
HSTN_SNVA    14022
SNVA_KSCY    13089
SNVA_HSTN     7676
SNVA_CHIN     5941
SNVA_ATLA     4283
SNVA_IPLS     3689
KSCY_SNVA     3686
SNVA_WASH     3590
KSCY_LOSA     3381
HSTN_LOSA     3316
dtype: int64


In [9]:
import pandas as pd
import numpy as np

ENTRADA   = "abilene_CORE_5min.csv"
INICIO    = "2004-05-01 00:00:00"      # tramo continuo sin cortes de medida
FIN       = "2004-08-19 23:55:00"
SAL_5MIN  = "abilene_FINAL_5min.csv"
SAL_1H    = "abilene_FINAL_1h.csv"

def calendario(df):
    ts = df["timestamp"]
    df["hour"]        = ts.dt.hour
    df["day_of_week"] = ts.dt.dayofweek
    df["hour_sin"]    = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"]    = np.cos(2 * np.pi * df["hour"] / 24)
    df["day_sin"]     = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["day_cos"]     = np.cos(2 * np.pi * df["day_of_week"] / 7)
    df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)
    return df

# 1. Recortar al tramo continuo
df = pd.read_csv(ENTRADA, parse_dates=["timestamp"])
df = df[(df["timestamp"] >= INICIO) & (df["timestamp"] <= FIN)].set_index("timestamp")
rejilla = pd.date_range(INICIO, FIN, freq="5min")
print(f"Instantes que faltan en el tramo: {len(rejilla.difference(df.index))}")   # debe ser 0

df = df.drop(columns=["ATLA_ATLA"])
# 2. NaN = tráfico casi nulo (demanda no anotada en el XML) → 0, igual que en BRAIN
print(f"NaN rellenados con 0: {df.isna().values.mean():.2%} de las celdas")
df = df.fillna(0)

# 3. Serie horaria: media de 12 valores de 5 min; etiqueta T = (T-60, T], igual que BRAIN
h = df.resample("1h", label="right", closed="right")
df1h = h.mean()[h.size() == 12]          # solo horas completas

# 4. Calendario y guardado
out5 = calendario(df.reset_index())
out1 = calendario(df1h.reset_index())
out5.to_csv(SAL_5MIN, index=False)
out1.to_csv(SAL_1H, index=False)

print(f"{SAL_5MIN}: {len(out5)} filas | {out5['timestamp'].min()} → {out5['timestamp'].max()}")
print(f"{SAL_1H}:   {len(out1)} filas | {out1['timestamp'].min()} → {out1['timestamp'].max()}")
print(f"Flujos: {df.shape[1]}")

Instantes que faltan en el tramo: 0
NaN rellenados con 0: 1.12% de las celdas


/tmp/ipykernel_34051/2669819345.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out5 = calendario(df.reset_index())
/tmp/ipykernel_34051/2669819345.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["hour"]        = ts.dt.hour
/tmp/ipykernel_34051/2669819345.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newfra

abilene_FINAL_5min.csv: 31968 filas | 2004-05-01 00:00:00 → 2004-08-19 23:55:00
abilene_FINAL_1h.csv:   2663 filas | 2004-05-01 01:00:00 → 2004-08-19 23:00:00
Flujos: 110


In [13]:
import pandas as pd
import numpy as np

def autopsia_csv(fichero):
    print(f"\n--- Autopsia de {fichero} ---")
    df = pd.read_csv(fichero)
    
    # 1. Buscar ceros reales
    total_ceros = (df == 0).sum().sum()
    print(f"Empty/Zero values: {total_ceros} ({total_ceros/df.size:.2%} del total)")
    
    # 2. Buscar outliers extremos (posibles errores de lectura)
    # Si el valor máximo es 1000 veces mayor que la mediana, hay algo raro.
    for col in df.select_dtypes(include=[np.number]).columns[:5]: # Solo las primeras 5 para no saturar
        mediana = df[col].median()
        maximo = df[col].max()
        if mediana > 0 and (maximo / mediana) > 1000:
            print(f"⚠️ Columna {col}: Máximo ({maximo}) es >1000x la mediana ({mediana}). Posible outlier.")

    # 3. Buscar columnas con varianza cero (datos muertos)
    constantes = [col for col in df.columns if df[col].nunique() <= 1]
    print(f"Columns with constant values: {len(constantes)}")
    if constantes:
        print(f"Ejemplos de columnas muertas: {constantes[:3]}")

autopsia_csv("abilene_FINAL_1h.csv")
autopsia_csv("abilene_FINAL_5min.csv")


--- Autopsia de abilene_FINAL_1h.csv ---
Empty/Zero values: 3008 (0.96% del total)
Columns with constant values: 0

--- Autopsia de abilene_FINAL_5min.csv ---
Empty/Zero values: 73951 (1.96% del total)
Columns with constant values: 0
